In [ ]:
import importlib, sys
import matplotlib as mpl
import numpy as np
import matplotlib.pyplot as plt
import random
import math
from math import hypot 
from openai import OpenAI 
import os
import concurrent.futures
import time
from collections import Counter
import re
from IPython import display
import ast
import csv
from datetime import datetime
from pathlib import Path
from dotenv import load_dotenv
import token
import requests
from prompt_toolkit import prompt
from dataclasses import dataclass
from typing import Optional, Tuple
import itertools
import json

In [ ]:
os.environ["OPENAI_API_KEY"] = "YOUR_OPENAI_API_KEY"

In [ ]:
client = OpenAI()

In [ ]:
# ============================================================
# ========  EXPERIMENT CONFIGURATION  ========================
# ============================================================

# --- Parameters to sweep ---
GRID_SIZES        = [20, 30, 50, 70]          # Map dimensions to test
AVG_SUGAR_VALUES  = [0.5, 1.0, 1.5, 2.0, 2.5, 3.0]  # Average sugar per cell (S_target = avg * GRID^2)

# --- Fixed parameters across all runs ---
NUM_AGENTS        = 50
MAX_STEPS         = 200
VISION_RANGE      = (1, 6)
METABOLISM_RANGE  = (0.5, 2.0)
INITIAL_SUGAR     = 15
ENABLE_HOUSES     = False
NUM_HOUSES        = 0

# --- Repetitions per configuration ---
N_REPEATS         = 1         # Number of runs per (grid, avg_sugar) combination
BASE_SEED         = 42        # Base seed; each repeat gets BASE_SEED + repeat_index

# --- Map generation ---
R_BASE = 6
W_BASE = 3

# --- Output ---
# All results go into a timestamped master folder under ./results/simulation_data
RESULTS_ROOT = Path("./results/simulation_data")

token_counter = {"prompt": 0, "completion": 0, "total": 0}

In [ ]:
# =========================================================
# ========= UTILITY / MAP FUNCTIONS =======================
# =========================================================
def shell_value(distance, radius):
    """Sugar core value: discrete levels 5->1"""
    if distance > radius:
        return 0
    if radius <= 0:
        return 5
    ring_width = radius / 5.0
    idx = int(min(4, math.floor(distance / max(ring_width, 1e-9))))
    return 5 - idx

def point_to_segment_distance(px, py, x1, y1, x2, y2):
    """Minimum distance from a point to a segment"""
    vx = x2 - x1
    vy = y2 - y1
    wx = px - x1
    wy = py - y1
    c1 = vx * wx + vy * wy
    if c1 <= 0:
        return hypot(px - x1, py - y1)
    c2 = vx * vx + vy * vy
    if c2 <= c1:
        return hypot(px - x2, py - y2)
    b = c1 / c2
    return hypot(px - (x1 + b * vx), py - (y1 + b * vy))

def corridor_value(dist, half_width):
    """Sugar value for corridors: discrete bands 3->1"""
    if dist > half_width:
        return 0
    if half_width <= 0:
        return 3
    band_width = half_width / 3.0
    idx = int(min(2, math.floor(dist / max(band_width, 1e-9))))
    return 3 - idx

def random_connections(num_nodes=4, max_deg=2, min_deg=1):
    """Generate random but valid connections"""
    edges = []
    degree = [0] * num_nodes
    possible_pairs = [(i, j) for i in range(num_nodes) for j in range(i + 1, num_nodes)]
    random.shuffle(possible_pairs)

    for node in range(num_nodes):
        while degree[node] < min_deg:
            candidates = [j for j in range(num_nodes)
                          if j != node and degree[j] < max_deg and (min(node, j), max(node, j)) not in edges]
            if not candidates:
                return edges
            partner = random.choice(candidates)
            edges.append((min(node, partner), max(node, partner)))
            degree[node] += 1
            degree[partner] += 1

    for a, b in possible_pairs:
        if degree[a] < max_deg and degree[b] < max_deg and (a, b) not in edges:
            edges.append((a, b))
            degree[a] += 1
            degree[b] += 1

    return edges

def get_centers(GRID):
    """Core positions scaled to grid size."""
    margin = max(5, GRID // 10)
    return [
        (margin, margin),
        (margin, GRID - margin - 1),
        (GRID - margin - 1, margin),
        (GRID - margin - 1, GRID - margin - 1),
    ]

def build_map_with_binary_search(GRID, centers, connections, S_target, r_base, w_base, tol=3, max_iter=40):
    """Binary search on scale factor to reach S_target"""
    lo, hi = 0.1, 3.0
    best_sugar_capacity = None
    
    for it in range(max_iter):
        m = (lo + hi) / 2.0
        radii = [max(1, int(round(r_base * m))) for _ in range(len(centers))]
        widths = {e: max(1, int(round(w_base * m))) for e in connections}
        
        sugar_capacity = np.zeros((GRID, GRID), dtype=float)
        
        # --- Core ---
        for i, (cx, cy) in enumerate(centers):
            r = radii[i]
            for xi in range(max(0, int(cx - r - 1)), min(GRID, int(cx + r + 2))):
                for yi in range(max(0, int(cy - r - 1)), min(GRID, int(cy + r + 2))):
                    d = hypot(xi - cx, yi - cy)
                    v = shell_value(d, r)
                    sugar_capacity[yi, xi] = max(sugar_capacity[yi, xi], v)
        
        # --- Corridors ---
        for (a, b) in connections:
            x1, y1 = centers[a]
            x2, y2 = centers[b]
            w = widths.get((a, b), widths.get((b, a), 1))
            half_w = w / 2.0
            for xi in range(GRID):
                for yi in range(GRID):
                    d = point_to_segment_distance(xi, yi, x1, y1, x2, y2)
                    v = corridor_value(d, half_w)
                    if v > 0:
                        sugar_capacity[yi, xi] = max(sugar_capacity[yi, xi], min(v, 3))
        
        sugar_capacity = np.clip(sugar_capacity, 0, 5)
        total_sugar = sugar_capacity.sum()
        
        if best_sugar_capacity is None or abs(total_sugar - S_target) < abs(best_sugar_capacity.sum() - S_target):
            best_sugar_capacity = sugar_capacity.copy()
        
        if abs(total_sugar - S_target) <= tol:
            break
        
        if total_sugar < S_target:
            lo = m
        else:
            hi = m
    
    return best_sugar_capacity


In [ ]:
# ============================================================
# ========  REGENERATION SUGAR  =============================
# ============================================================
def regenerate_sugar(sugar, sugar_capacity, houses, step):
    if step % 2 == 0:
        if ENABLE_HOUSES:
            can_regen = (sugar_capacity > 0) & (~houses)
        else:
            can_regen = (sugar_capacity > 0)
        sugar[can_regen] = np.minimum(sugar[can_regen] + 1, sugar_capacity[can_regen])
    return sugar

In [ ]:
# =========================================================
# ========= AGENT CLASS ===================================
# =========================================================

class LMAgent:
    _prompt_logged = False  # Class-level flag

    def __init__(self, x, y, vision, metabolism, sugar, memory_length=50):
        self.x = x
        self.y = y
        self.vision = vision
        self.metabolism = metabolism
        self.sugar = sugar
        self.alive = True
        self.memory = []
        self.memory_length = memory_length
        self.age = 0
        self.alpha = random.uniform(-1, 1)  
        self.in_house = False
        self.target = None 
        self.last_collected = 0.0
        self.id = 0
        self.last_decision_str = ""
        
    def perceive(self, sugar_map, agents, pos_to_agent):
        perceived = []
        for dx in range(-self.vision, self.vision + 1):
            for dy in range(-self.vision, self.vision + 1):
                # Circular vision <= vision
                if dx == 0 and dy == 0:
                    continue  # Exclude current cell
                if math.hypot(dx, dy) <= self.vision:
                    nx, ny = self.x + dx, self.y + dy
                    if 0 <= nx < sugar_map.shape[1] and 0 <= ny < sugar_map.shape[0]:
                        cell_sugar = sugar_map[ny, nx]
                        occupied = (nx, ny) in pos_to_agent
                        perceived.append({'x': nx, 'y': ny, 'sugar': cell_sugar, 'occupied': occupied})
        return perceived

    def get_decision(self, sugar_map, houses, vision_cells, string_vision_cells,
                      am_i_in_house, string_houses_seen, occupied_neighbors=None, 
                      alfa_neighbors=None, vision_neighbors=None):
        
        system_prompt = f"""# SUGARSCAPE AGENT
        
        ## RULES
        - can eat sugar from the cell it is on
        - has metabolism cost per turn (half if in house)
        - can see cells within vision range (circular) 
        - only attack on adjacent cells
        - you move once per turn, so moving to a cell distant n movements takes n turns

        ## ATTACKS
        - identity genes are between -1 and 1: the more your is different from the surrounding agents, the more prone you are to attack and the more you are likely to be attacked    
        - if you WIN an ATTACK, steal all sugar from target
        - if you LOSE an ATTACK, lose all your sugar to target and DIE

        ## REPRODUCE
        - create another agent with similar identity to yours
        - costs half your sugar

        ## INPUT FORMAT: visible cells are dict with x=column, y=row, z=sugar

        ## FOUR ACTIONS AVAILABLE
        1,(x,y) --> MOVE to visible cell
        2 --> STAY 
        3,(x,y) --> REPRODUCE at adjacent empty cell (requires 30+ sugar, costs half)
        4,(x,y) --> ATTACK adjacent agent (to steal all sugar, risk death)

        ## RESPOND WITH ONLY: N or N,(x,y) — Example: 1,(5,3) or 2 or 3,(4,6) or 4,(2,7)"""
        
        memory_str = "\n".join([str(obs) for obs in self.memory]) if self.memory else "(no memory)"
        current_cell = {'x': self.x, 'y': self.y, 'z': int(sugar_map[self.y, self.x])}

        move_prompt = f"""Current state: sugar = {self.sugar}, metabolism = {self.metabolism}, identity gene = {self.alpha:.2f}.
        Your current position is ({current_cell}) --> on a house? {am_i_in_house}.
        Agent memory of the last {self.memory_length} events:\n{memory_str}\n
        \nAvailable sugar in visible cells (list of dicts): {string_vision_cells}
        Houses seen in your vision: {string_houses_seen}."""
        
         # Agenti visibili (tutti quelli in visione)
        agents_prompt = ""
        if vision_neighbors:
            agents_prompt = f"\nAGENTS IN VISION: {vision_neighbors}"

        attack_prompt = ""
        if occupied_neighbors is not None and alfa_neighbors is not None:
            attack_prompt = f"""\nADJACENT AGENTS (attackable): {occupied_neighbors}.
                \nAlpha values of adjacent agents are: {alfa_neighbors}."""

        prompt = move_prompt + agents_prompt + attack_prompt

        t0 = time.time()
        #print(f"[{time.strftime('%H:%M:%S')}] Agent {self.id} START")

        answer = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content":prompt},
            ],
            max_tokens=20,
            temperature=0.7,
        )
        #print(f"[{time.strftime('%H:%M:%S')}] Agent {self.id} END - {time.time()-t0:.2f}s"
        # Registra i token
        usage = answer.usage
        token_counter["prompt"]     += usage.prompt_tokens
        token_counter["completion"] += usage.completion_tokens
        token_counter["total"]      += usage.total_tokens

        return answer.choices[0].message.content.strip()
    
    def memory_append(self, observation):
        self.memory.append(observation)
        if len(self.memory) > self.memory_length:
            self.memory.pop(0)


In [ ]:
#=========================================================
# ======== DECISION PARSING ==============================
#=========================================================
@dataclass
class AgentDecision:
    action: int          # 1=move, 2=stay, 3=reproduce, 4=attack
    target: Optional[Tuple[int, int]] = None
    raw: str = ""
    valid: bool = True

def parse_decision(raw: str, agent_x: int, agent_y: int, grid_size: int) -> AgentDecision:
    """Parse LLM output into a structured decision. Falls back to STAY on any error."""
    raw = raw.strip()
    
    # Trova il tipo di azione (primo digit)
    action_match = re.match(r'(\d)', raw)
    if not action_match:
        return AgentDecision(action=2, raw=raw, valid=False)
    
    action = int(action_match.group(1))
    
    if action not in (1, 2, 3, 4):
        return AgentDecision(action=2, raw=raw, valid=False)
    
    # Azione 2: nessuna coordinata necessaria
    if action == 2:
        return AgentDecision(action=2, raw=raw)
    
    # Per azioni 1, 3, 4: estrai coordinate (x, y)
    coord_match = re.search(r'\((\d+)\s*,\s*(\d+)\)', raw)
    if not coord_match:
        return AgentDecision(action=2, raw=raw, valid=False)
    
    tx, ty = int(coord_match.group(1)), int(coord_match.group(2))
    
    # Validazione bounds
    if not (0 <= tx < grid_size and 0 <= ty < grid_size):
        return AgentDecision(action=2, raw=raw, valid=False)
    
    # Validazione adiacenza per attacco e riproduzione
    if action in (3, 4):
        if abs(tx - agent_x) > 1 or abs(ty - agent_y) > 1:
            return AgentDecision(action=2, raw=raw, valid=False)
    
    return AgentDecision(action=action, target=(tx, ty), raw=raw)

In [ ]:
#=========================================================
# ======== LOGGING FUNCTIONS =============================
#=========================================================

def create_run_folder(batch_folder, grid, avg_sugar, repeat):
    """Create subfolder for a single run."""
    run_name = f"G{grid}_S{avg_sugar:.2f}_R{repeat}"
    run_path = batch_folder / run_name
    run_path.mkdir(parents=True, exist_ok=True)
    return run_path

def init_csv_files(folder_path):
    moves_file = folder_path / "agent_moves.csv"
    with open(moves_file, 'w', newline='', encoding='utf-8') as f:
        csv.writer(f).writerow([
            'step', 'agent_id', 'alpha', 'x', 'y', 'sugar', 'metabolism', 'age',
            'decision', 'target_x', 'target_y', 'target_alpha', 'collected', 'in_house', 'alive'
        ])
    stats_file = folder_path / "step_stats.csv"
    with open(stats_file, 'w', newline='', encoding='utf-8') as f:
        csv.writer(f).writerow([
            'step', 'agents_alive', 'total_map_sugar', 'total_agent_sugar',
            'combats_total', 'avg_agent_sugar', 'avg_agent_age', 'births_total'
        ])
    map_file = folder_path / "map_state.csv"
    with open(map_file, 'w', newline='', encoding='utf-8') as f:
        csv.writer(f).writerow(['step', 'x', 'y', 'sugar', 'is_house'])
    return moves_file, stats_file, map_file

def log_agent_move(moves_file, step, agent, decision_str, target_alpha=None):
    target_x, target_y = agent.target if agent.target else (None, None)
    with open(moves_file, 'a', newline='', encoding='utf-8') as f:
        csv.writer(f).writerow([
            step, agent.id, round(agent.alpha, 3), agent.x, agent.y,
            round(agent.sugar, 1), agent.metabolism, agent.age, decision_str,
            target_x, target_y, round(target_alpha, 3) if target_alpha else None,
            round(agent.last_collected, 1), agent.in_house, agent.alive
        ])

def log_step_stats(stats_file, step, agent_list, sugar_map, combat_counter, birth_counter=0):
    alive = [a for a in agent_list if a.alive]
    total_agent_sugar = sum(a.sugar for a in alive) if alive else 0
    avg_sugar = total_agent_sugar / len(alive) if alive else 0
    avg_age = sum(a.age for a in alive) / len(alive) if alive else 0
    with open(stats_file, 'a', newline='', encoding='utf-8') as f:
        csv.writer(f).writerow([
            step, len(alive), int(sugar_map.sum()), round(total_agent_sugar, 1),
            combat_counter[0], round(avg_sugar, 2), round(avg_age, 2), birth_counter
        ])

def log_map_state(map_file, step, sugar_map, houses, agent_list):
    agents_in_houses = {(a.x, a.y) for a in agent_list if a.alive and a.in_house}
    with open(map_file, 'a', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        GRID = sugar_map.shape[0]
        for y in range(GRID):
            for x in range(GRID):
                sugar_val = sugar_map[y, x]
                is_house = houses[y, x]
                if sugar_val > 0 or is_house:
                    has_agent = (x, y) in agents_in_houses
                    house_state = 0
                    if is_house:
                        house_state = 2 if has_agent else 1
                    writer.writerow([step, x, y, int(sugar_val), house_state])

def save_run_config(folder_path, seed, config):
    """Save full run configuration."""
    config_file = folder_path / "config.csv"
    with open(config_file, 'w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow(['parameter', 'value'])
        for k, v in config.items():
            writer.writerow([k, v])

def save_initial_agents(folder_path, agent_list):
    initial_file = folder_path / "initial_agents.csv"
    with open(initial_file, 'w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow(['id', 'x', 'y', 'vision', 'metabolism', 'sugar', 'alpha'])
        for a in agent_list:
            writer.writerow([a.id, a.x, a.y, a.vision, a.metabolism, round(a.sugar, 1), round(a.alpha, 3)])

def save_final_summary(folder_path, agent_list, sugar_map, combat_counter, steps_run, num_agents_initial):
    alive = [a for a in agent_list if a.alive]
    summary_file = folder_path / "final_summary.csv"
    with open(summary_file, 'w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow(['metric', 'value'])
        writer.writerow(['total_steps', steps_run])
        writer.writerow(['agents_initial', num_agents_initial])
        writer.writerow(['agents_surviving', len(alive)])
        writer.writerow(['total_combats', combat_counter[0]])
        writer.writerow(['final_map_sugar', int(sugar_map.sum())])
        if alive:
            writer.writerow(['avg_final_sugar', round(sum(a.sugar for a in alive) / len(alive), 2)])
            writer.writerow(['avg_final_age', round(sum(a.age for a in alive) / len(alive), 2)])
            writer.writerow(['max_sugar', round(max(a.sugar for a in alive), 1)])
            writer.writerow(['max_age', max(a.age for a in alive)])
            
    # 2. Salva il dettaglio dei sopravvissuti (Parte AGGIUNTA per completezza totale)
    if alive:
        survivors_file = folder_path / "survivors.csv"
        with open(survivors_file, 'w', newline='', encoding='utf-8') as f:
            writer = csv.writer(f)
            writer.writerow(['id', 'alpha', 'sugar', 'age', 'vision', 'metabolism', 'x', 'y'])
            for a in alive:
                writer.writerow([a.id, round(a.alpha, 3), round(a.sugar, 1), a.age, a.vision, a.metabolism, a.x, a.y])

def save_houses_config(folder_path, houses):
    """Save house positions for replay"""
    houses_file = folder_path / "houses.csv"
    with open(houses_file, 'w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow(['x', 'y'])
        GRID = houses.shape[0]
        for y in range(GRID):
            for x in range(GRID):
                if houses[y, x]:
                    writer.writerow([x, y])
    print(f"✓ Houses saved: {houses.sum()} houses")

def save_sugar_capacity(folder_path, sugar_capacity):
    """Save sugar capacity map for replay"""
    capacity_file = folder_path / "sugar_capacity.csv"
    with open(capacity_file, 'w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow(['x', 'y', 'capacity'])
        GRID = sugar_capacity.shape[0]
        for y in range(GRID):
            for x in range(GRID):
                if sugar_capacity[y, x] > 0:
                    writer.writerow([x, y, int(sugar_capacity[y, x])])
    print(f"✓ Sugar capacity saved")
    

In [ ]:
# ============================================================
# ========  REPRODUCE AGENT  =================================
# ============================================================

def reproduce_agent(agent, agent_list, houses, GRID, child_x, child_y, next_id, birth_counter):
    # Create the child only if the cell is free and not a house
    if not any(a.x == child_x and a.y == child_y and a.alive for a in agent_list) and not houses[child_y, child_x]:
        vision = agent.vision #max(1, min(agent.vision + np.random.choice([-1,0,1]), 6))
        metabolism = agent.metabolism #max(1, min(agent.metabolism + np.random.choice([-1,0,1]), 4))
        sugar = agent.sugar // 2
        alpha = np.clip(agent.alpha + np.random.normal(0, 0.05), -1, 1)
        child = LMAgent(child_x, child_y, vision, metabolism, sugar)
        child.alpha = alpha
        child.id = next_id
        agent.sugar -= sugar
        agent.memory_append(f"REPR: {child.id}@({child_x},{child_y})")
        birth_counter[0] += 1
        return child
    return None

In [ ]:
def debug_agent_move(agent, decision, attack_res, result_msg):
    # Determiniamo lo stato del movimento
    if getattr(agent, 'target', None) is not None:
        status_movimento = f"IN VIAGGIO verso {agent.target}"
    else:
        status_movimento = "OBIETTIVO RAGGIUNTO / IN ATTESA"

    print(f"\n--- DEBUG AGENTE {agent.id} (Alfa: {agent.alpha:.2f}) ---")
    print(f"  STATO: {status_movimento}")
    print(f"  POSIZIONE: ({agent.x}, {agent.y})")
    print(f"  ZUCCHERO POSSEDUTO: {agent.sugar:.1f}  <--")
    print(f"  METABOLISMO: -{agent.metabolism}")
    print(f"  VISIONE: {agent.vision}")
    print(f"  ETÀ: {agent.age}")
    color_sugar = "+" if agent.last_collected > 0 else ""
    print(f"  ZUCCHERO RACCOLTO: {color_sugar}{agent.last_collected:.1f}")
    
    if attack_res != "-1":
        print(f"  SCELTA ATTACCO: Ha provato ad attaccare {attack_res}")
    else:
        print(f"  SCELTA ATTACCO: Nessun attacco")
        
    # Mostriamo la decisione dell'LLM solo se è stata presa in questo turno
    # Se decision è None o vuota, significa che l'agente sta usando una vecchia decisione
    if decision:
        print(f"  NUOVA DECISIONE LLM: {decision}")
    else:
        # Se l'agente sta viaggiando, mostriamo l'azione che intende fare all'arrivo
        azione_pianificata = getattr(agent, 'last_decision_str', 'Nessuna')
        print(f"  AZIONE PIANIFICATA ALL'ARRIVO: {azione_pianificata}")

    print(f"  ULTIMO LOG MEMORIA: {agent.memory[-1] if agent.memory else 'Nessun log'}")
    print("-" * 35)

In [ ]:
# ============================================================
# ========  PROCESS TURN  ====================================
# ============================================================

def process_lma_turn(agent_list, sugar_map, GRID, houses, combat_counter, birth_counter, step=0, moves_file=None):
    alive_agents = [a for a in agent_list if a.alive]
    turn_data = []
    
    # Lookup O(1) per posizione
    pos_to_agent = {(a.x, a.y): a for a in alive_agents}

    for agent in alive_agents:
        if not agent.alive:
            continue
           
        # 1. PERCEPTION
        perceived = agent.perceive(sugar_map, agent_list, pos_to_agent)

        # All free cells in vision
        vision_cells = [
            {'x': p['x'], 'y': p['y'], 'z': int(p['sugar'])}
            for p in perceived
            if not p['occupied'] and (p['x'], p['y']) != (agent.x, agent.y)
        ]
        string_vision_cells = str(vision_cells)

        # Alive agents in vision (excluding self)
        vision_neighbors = [
            {'x': p['x'], 'y': p['y'], 'alpha': pos_to_agent[(p['x'], p['y'])].alpha}
            for p in perceived if p['occupied'] and (p['x'], p['y']) != (agent.x, agent.y)
        ]

        if ENABLE_HOUSES:
            # Find houses within vision range
            houses_in_vision = []
            for dx in range(-agent.vision, agent.vision + 1):
                for dy in range(-agent.vision, agent.vision + 1):
                    if dx == 0 and dy == 0:
                        continue
                    if math.hypot(dx, dy) <= agent.vision:  
                        nx, ny = agent.x + dx, agent.y + dy
                        if 0 <= nx < GRID and 0 <= ny < GRID:
                            if houses[ny, nx]:
                                houses_in_vision.append({'x': nx, 'y': ny})
            
            # Check if the current cell is a house
            am_i_in_house = "YES" if houses[agent.y, agent.x] else "NO"
            if houses_in_vision:
                string_houses_seen = str(houses_in_vision)
            else:
                string_houses_seen = "None"

            # If houses are found, add them to memory as "observation"
            if houses_in_vision:
                    coords = ",".join([f"({h['x']},{h['y']})" for h in houses_in_vision])
                    agent.memory_append(f"CASE:{coords}")
        
        else:
            am_i_in_house = "NO"
            string_houses_seen = "None"

        # Identify neighbors (adjacent) for the attack prompt
        # Filter those at distance 1 and do not consider those in house
        adj_info = []
        for p in perceived:
            if p['occupied']:
                target_a = pos_to_agent.get((p['x'], p['y']))
                if target_a and not target_a.in_house and (p['x'], p['y']) != (agent.x, agent.y):
                    if abs(p['x'] - agent.x) <= 1 and abs(p['y'] - agent.y) <= 1:
                        adj_info.append(p)
            
        occupied_neighbors = [
            {'x': p['x'], 'y': p['y'], 'alpha': next(a['alpha'] for a in vision_neighbors if a['x'] == p['x'] and a['y'] == p['y'])}
            for p in adj_info
        ]
        alfa_neighbors = [p['alpha'] for p in vision_neighbors]

        turn_data.append({
            'agent': agent,
            'am_i_in_house': am_i_in_house,
            'string_houses_seen': string_houses_seen,
            'occupied_neighbors': occupied_neighbors,
            'vision_neighbors': vision_neighbors,
            'alfa_neighbors': alfa_neighbors,
            'vision_cells': vision_cells,
            'string_vision_cells': string_vision_cells
        })

    # --- 2. THINKING PHASE (Parallel) ---
    agents_needing_decision = [d for d in turn_data if d['agent'].target is None]
    decision = {}
    if agents_needing_decision:

        # Limita a 20-30 per evitare rate limits e overhead
        max_parallel = min(50, len(agents_needing_decision))
        with concurrent.futures.ThreadPoolExecutor(max_workers=max_parallel) as executor:
            future_to_agent = {
                executor.submit(
                    d['agent'].get_decision, 
                    sugar_map,
                    houses,
                    d['vision_cells'],
                    d['string_vision_cells'], 
                    d['am_i_in_house'],
                    d['string_houses_seen'],
                    d['occupied_neighbors'],
                    d['alfa_neighbors'],
                    d['vision_neighbors']
                ): d['agent'] 
                for d in agents_needing_decision
            }
            for f in concurrent.futures.as_completed(future_to_agent):
                decision[future_to_agent[f]] = f.result()

    # --- 3. EXECUTION PHASE (Sequential) ---
    for d in turn_data:
        agent = d['agent']
        if not agent.alive: 
            print(f"Agent {agent.id} died at the start of execution.")
            continue
        
        agent.last_collected = 0  # Reset collection for this turn
        attack_target_alpha = None
        # Execute Attack if decided
        decision_str = decision.get(agent)
        # Track action for logging
        action_log = decision_str if decision_str else "TRAVELING" if agent.target else "NO_NEW_DEC"

        # If there is a new decision, process it
        if decision_str:
            agent.memory_append(f"DEC: {decision_str}")
            dec= parse_decision(decision_str, agent.x, agent.y, GRID)

            # ACTION 4: ATTACK
            if dec.action == 4 and dec.target:
                target_x, target_y = dec.target
                target_agent = pos_to_agent.get((target_x, target_y))
                
                if target_agent and not target_agent.in_house:
                    attack_target_alpha = target_agent.alpha
                    if agent.in_house:
                        agent.in_house = False 
                    combat_counter[0] += 1
                    power_a = agent.sugar + random.random()
                    power_b = target_agent.sugar + random.random()
                    
                    if power_a > power_b:
                        stolen = target_agent.sugar
                        agent.sugar += stolen
                        target_agent.sugar = 0
                        target_agent.alive = False
                        agent.memory_append(f"ATK WIN: +{int(stolen)} da ({target_x},{target_y})")
                        action_log = f"4,({target_x},{target_y})-WIN"
                        pos_to_agent.pop((target_x, target_y), None)
                    else:
                        stolen = agent.sugar
                        target_agent.sugar += stolen
                        agent.sugar = 0
                        agent.alive = False
                        target_agent.memory_append(f"DEF WIN: +{int(stolen)} da ({agent.x},{agent.y})")
                        action_log = f"4,({target_x},{target_y})-LOSE"
                        pos_to_agent.pop((agent.x, agent.y), None)
                else:
                    agent.memory_append("ATK FAIL: invalid target")
                    action_log = f"4,({target_x},{target_y})-FAIL"

            # ACTION 1: MOVE
            elif dec.action == 1 and dec.target:
                agent.target = dec.target
                agent.last_decision_str = decision_str
                action_log = f"1,({dec.target[0]},{dec.target[1]})-START"
            
            # ACTION 2: STAY STILL
            elif dec.action == 2:
                agent.target = None
                action_log = "2-STAY"

            # ACTION 3: REPRODUCE (logica spostata qui)
            elif dec.action == 3 and dec.target:
                agent.target = None
                sugar_threshold = 30
                child_x, child_y = dec.target
                
                if agent.sugar >= sugar_threshold:
                    next_id = max(a.id for a in agent_list) + 1 if agent_list else 0
                    child = reproduce_agent(agent, agent_list, houses, GRID, child_x, child_y, next_id, birth_counter)
                    if child:
                        agent_list.append(child)
                        action_log = f"3,({child_x},{child_y})-SUCCESS"
                    else:
                        agent.memory_append("REPR: FAIL-OCC")
                        action_log = f"3,({child_x},{child_y})-FAIL-OCC"
                else:
                    agent.memory_append("REPR: FAIL-LOW SUG")
                    action_log = f"3,({child_x},{child_y})-FAIL-LOW SUG"

            # FALLBACK
            else:
                agent.target = None
                agent.memory_append(f"ERR: UNK-{dec.action}")
                action_log = decision_str    

        if not agent.alive:
            if moves_file:
                log_agent_move(moves_file, step, agent, decision_str or "DIED", target_alpha=attack_target_alpha)
            continue

        # Move the agent towards the target (if any)
        if getattr(agent, 'target', None):
            tx, ty = agent.target
            if (agent.x, agent.y) == (tx, ty):
                # Arrived: execute the final action (e.g., collect or build)
                agent.target = None
                agent.memory_append(f"ARR: ({tx}, {ty})")
                # COLLECTION UPON ARRIVAL (to not lose the sugar on the target cell)
                available = sugar_map[agent.y, agent.x]
                sugar_map[agent.y, agent.x] = 0
                agent.sugar += available
                agent.last_collected = available
                agent.in_house = houses[agent.y, agent.x]

            else:
                # Take a step
                adj_cells = [(nx, ny) for nx, ny in [(agent.x+1, agent.y), (agent.x-1, agent.y), (agent.x, agent.y+1), (agent.x, agent.y-1)]
                             if 0 <= nx < GRID and 0 <= ny < GRID and (nx, ny) not in pos_to_agent]
                
                if adj_cells:
                    old_x, old_y = agent.x, agent.y  # Per logging del movimento
                    next_step = min(adj_cells, key=lambda c: abs(c[0]-tx) + abs(c[1]-ty))
                    agent.x, agent.y = next_step
                    pos_to_agent.pop((old_x, old_y), None)  # Rimuovi vecchia posizione
                    pos_to_agent[(agent.x, agent.y)] = agent  # Aggiorna nuova posizione

                    # Automatic collection during the journey
                    available = sugar_map[agent.y, agent.x]
                    sugar_map[agent.y, agent.x] = 0
                    agent.last_collected = available
                    agent.sugar += available
                    agent.in_house = houses[agent.y, agent.x]
                    if (agent.x, agent.y) == (tx, ty): agent.target = None
                    agent.memory_append(f"MOV: ({agent.x}, {agent.y})→({tx}, {ty}) + {available}.")
                else:
                    agent.memory_append(f"BLOCK-> ({tx}, {ty})")
            
        # --- STATIONARY COLLECTION (If not moved) ---
        # If the agent is stationary, check if sugar has regrown beneath them
        if getattr(agent, 'target', None) is None:
            available = sugar_map[agent.y, agent.x]
            if available > 0:
                sugar_map[agent.y, agent.x] = 0
                agent.last_collected = available
                agent.sugar += available
                agent.memory_append(f"STAY+{available}")

        # --- METABOLISM CONSUMPTION LOGIC ---
        if ENABLE_HOUSES and agent.in_house and houses[agent.y, agent.x]:
            cost = max(agent.metabolism / 2, 1)
            tipo_loc = "in house"
        else:
            agent.in_house = False 
            cost = agent.metabolism
            tipo_loc = "outside"

        # Apply the cost ONLY ONCE per turn
        agent.sugar -= cost
        agent.age += 1
        agent.memory_append(f"MET ({tipo_loc}): -{cost}")

        if agent.sugar <= 0:
            agent.sugar = 0
            agent.alive = False
            agent.memory_append("DEAD.")

        # --- LOGGING THE MOVE ---
        if moves_file:
            log_agent_move(moves_file, step, agent, action_log, target_alpha=attack_target_alpha)


In [ ]:

# ============================================================
# ========  SINGLE SIMULATION RUN  ===========================
# ============================================================

def run_single_simulation(grid, avg_sugar, repeat_idx, batch_folder, base_seed):
    """Run one simulation with given parameters. Returns a summary dict."""

    seed = base_seed + repeat_idx
    random.seed(seed)
    np.random.seed(seed)

    run_vision = random.randint(*VISION_RANGE)
    run_metabolism = random.uniform(*METABOLISM_RANGE)

    GRID = grid
    S_target = int(round(avg_sugar * GRID * GRID))

    # Build map
    centers = get_centers(GRID)
    connections = random_connections(num_nodes=len(centers), max_deg=2)

    sugar_capacity = build_map_with_binary_search(
        GRID, centers, connections, S_target, R_BASE, W_BASE, tol=3, max_iter=40
    )
    sugar_capacity = np.rint(sugar_capacity).astype(int)
    actual_total = int(sugar_capacity.sum())
    actual_avg = actual_total / (GRID * GRID)

    houses = np.zeros((GRID, GRID), dtype=bool)
    sugar_map = sugar_capacity.copy().astype(int)

    # Create agents
    agent_list = []
    occupied = set()
    for i in range(NUM_AGENTS):
        while True:
            x = random.randint(0, GRID - 1)
            y = random.randint(0, GRID - 1)
            if (x, y) not in occupied:
                occupied.add((x, y))
                break
        agent = LMAgent(x, y, run_vision, run_metabolism, INITIAL_SUGAR)
        agent.id = i
        agent_list.append(agent)

    initial_count = len(agent_list)

    # Setup logging
    run_folder = create_run_folder(batch_folder, GRID, avg_sugar, repeat_idx)
    moves_file, stats_file, map_file = init_csv_files(run_folder)

    config = {
        'seed': seed,
        'grid_size': GRID,
        'avg_sugar_target': avg_sugar,
        'S_target': S_target,
        'actual_total_sugar': actual_total,
        'actual_avg_sugar': round(actual_avg, 4),
        'num_agents': NUM_AGENTS,
        'max_steps': MAX_STEPS,
        'global_vision': run_vision,
        'global_metabolism': run_metabolism,
        'initial_sugar': INITIAL_SUGAR,
        'repeat_index': repeat_idx,
    }
    save_run_config(run_folder, seed, config)
    save_initial_agents(run_folder, agent_list)
    save_houses_config(run_folder, houses)
    save_sugar_capacity(run_folder, sugar_capacity)

    combat_counter = [0]
    birth_counter = [0]

    print(f"\n  ▶ RUN: Grid={GRID}, AvgSugar={avg_sugar:.2f} (actual={actual_avg:.3f}), "
          f"Repeat={repeat_idx}, Seed={seed}")

    # Simulation loop
    steps_run = 0
    for step in range(MAX_STEPS):
        steps_run = step + 1
        random.shuffle(agent_list)
        process_lma_turn(agent_list, sugar_map, GRID, houses, combat_counter, birth_counter, step, moves_file)
        sugar_map = regenerate_sugar(sugar_map, sugar_capacity, houses, step)
        agent_list[:] = [a for a in agent_list if a.alive]
        log_step_stats(stats_file, step, agent_list, sugar_map, combat_counter, birth_counter=birth_counter[0])
        log_map_state(map_file, step, sugar_map, houses, agent_list)

        if step % 10 == 0:
            print(f"    Step {step:3d} | Alive: {len(agent_list):2d} | "
                  f"MapSugar: {int(sugar_map.sum()):5d} | Combats: {combat_counter[0]} | Births: {birth_counter[0]} | "
                  f"Tokens: {token_counter['total']:,} (prompt={token_counter['prompt']:,}, completion={token_counter['completion']:,})")

        if not agent_list:
            print(f"    ⚠ All agents dead at step {step}")
            break

    save_final_summary(run_folder, agent_list, sugar_map, combat_counter, steps_run, initial_count)

    alive = [a for a in agent_list if a.alive]
    summary = {
        'grid_size': GRID,
        'avg_sugar_target': avg_sugar,
        'global_vision': run_vision,
        'global_metabolism': run_metabolism,
        'actual_avg_sugar': round(actual_avg, 4),
        'actual_total_sugar': actual_total,
        'repeat': repeat_idx,
        'seed': seed,
        'steps_run': steps_run,
        'agents_initial': initial_count,
        'agents_surviving': len(alive),
        'survival_rate': round(len(alive) / initial_count, 4) if initial_count > 0 else 0,
        'total_combats': combat_counter[0],
        'total_births': birth_counter[0],
        'final_map_sugar': int(sugar_map.sum()),
        'avg_final_agent_sugar': round(sum(a.sugar for a in alive) / len(alive), 2) if alive else 0,
        'avg_final_age': round(sum(a.age for a in alive) / len(alive), 2) if alive else 0,
        'run_folder': str(run_folder),
    }

    print(f"    ✅ Done. Surviving: {len(alive)}/{initial_count}")
    return summary

In [ ]:
# ============================================================
# ========  BATCH RUNNER  ====================================
# ============================================================

def run_batch():
    """Run all parameter combinations and collect results."""

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    batch_folder = RESULTS_ROOT / f"batch_{timestamp}"
    batch_folder.mkdir(parents=True, exist_ok=True)

    # Save batch-level config
    batch_config = {
        'grid_sizes': GRID_SIZES,
        'avg_sugar_values': AVG_SUGAR_VALUES,
        'n_repeats': N_REPEATS,
        'base_seed': BASE_SEED,
        'num_agents': NUM_AGENTS,
        'max_steps': MAX_STEPS,
        'vision_range': VISION_RANGE,
        'metabolism_range': METABOLISM_RANGE,
        'initial_sugar': INITIAL_SUGAR,
        'enable_houses': ENABLE_HOUSES,
    }
    with open(batch_folder / "batch_config.json", 'w') as f:
        json.dump(batch_config, f, indent=2)

    # Generate all combinations
    combinations = list(itertools.product(GRID_SIZES, AVG_SUGAR_VALUES, range(N_REPEATS)))
    total_runs = len(combinations)

    print("=" * 60)
    print(f"  BATCH SIMULATION")
    print(f"  Grid sizes:       {GRID_SIZES}")
    print(f"  Avg sugar values: {AVG_SUGAR_VALUES}")
    print(f"  Repeats per combo: {N_REPEATS}")
    print(f"  Total runs:       {total_runs}")
    print(f"  Output folder:    {batch_folder}")
    print("=" * 60)

    all_summaries = []
    start_time = time.time()

    for run_idx, (grid, avg_sugar, repeat) in enumerate(combinations):
        print(f"\n{'─'*50}")
        print(f"  Run {run_idx + 1}/{total_runs}")

        try:
            summary = run_single_simulation(grid, avg_sugar, repeat, batch_folder, BASE_SEED)
            all_summaries.append(summary)
        except Exception as e:
            print(f"    ❌ ERROR: {e}")
            all_summaries.append({
                'grid_size': grid,
                'avg_sugar_target': avg_sugar,
                'repeat': repeat,
                'error': str(e),
            })

    elapsed = time.time() - start_time

    # ---- Save master summary CSV ----
    master_file = batch_folder / "master_summary.csv"
    if all_summaries:
        keys = all_summaries[0].keys()
        with open(master_file, 'w', newline='', encoding='utf-8') as f:
            writer = csv.DictWriter(f, fieldnames=keys)
            writer.writeheader()
            # Sostituisci la riga 69 con:
            valid_summaries = [s for s in all_summaries if 'error' not in s]
            if valid_summaries:
                keys = valid_summaries[0].keys()
                with open(master_file, 'w', newline='', encoding='utf-8') as f:
                    writer = csv.DictWriter(f, fieldnames=keys)
                    writer.writeheader()
                    writer.writerows(valid_summaries)

    # ---- Save aggregated stats (mean across repeats) ----
    agg_file = batch_folder / "aggregated_stats.csv"
    with open(agg_file, 'w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow([
            'grid_size', 'avg_sugar_target', 'actual_avg_sugar_mean',
            'survival_rate_mean', 'survival_rate_std',
            'combats_mean', 'births_mean',
            'avg_final_sugar_mean', 'avg_final_age_mean',
            'n_runs'
        ])

        for grid in GRID_SIZES:
            for avg_sugar in AVG_SUGAR_VALUES:
                group = [s for s in all_summaries
                         if s.get('grid_size') == grid
                         and s.get('avg_sugar_target') == avg_sugar
                         and 'error' not in s]
                if not group:
                    continue
                n = len(group)
                sr = [s['survival_rate'] for s in group]
                writer.writerow([
                    grid, avg_sugar,
                    round(np.mean([s['actual_avg_sugar'] for s in group]), 4),
                    round(np.mean(sr), 4),
                    round(np.std(sr), 4),
                    round(np.mean([s['total_combats'] for s in group]), 2),
                    round(np.mean([s['total_births'] for s in group]), 2),
                    round(np.mean([s['avg_final_agent_sugar'] for s in group]), 2),
                    round(np.mean([s['avg_final_age'] for s in group]), 2),
                    n,
                ])

    print("\n" + "=" * 60)
    print(f"  BATCH COMPLETE")
    print(f"  Total time:    {elapsed:.1f}s ({elapsed/60:.1f}min)")
    print(f"  Runs:          {total_runs}")
    print(f"  Results in:    {batch_folder}")
    print(f"  Master CSV:    {master_file}")
    print(f"  Aggregated:    {agg_file}")
    print("=" * 60)

    return batch_folder

In [ ]:
run_batch()